# 04 简单推荐系统（协同过滤）\n\n## 分析目标\n- 基于用户-物品交互矩阵，实现基于用户的协同过滤（UserCF）\n- 计算用户相似度矩阵，为指定用户生成推荐列表\n- 使用准确率与召回率评估推荐效果

In [ ]:
import pandas as pd\nimport numpy as np\nimport matplotlib.pyplot as plt\nimport seaborn as sns\nimport warnings\nimport os\nfrom datetime import datetime, timedelta\nfrom collections import defaultdict\nimport sys
sys.path.insert(0, '..')
\nwarnings.filterwarnings('ignore')\nplt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei']\nplt.rcParams['axes.unicode_minus'] = False\nsns.set_style('whitegrid')\n\nfrom config import IMAGES_DIR as IMG_DIR
\nprint('环境初始化完成')

## 1. 数据加载

In [ ]:
from config import CLEANED_CSV_PATH as DATA_PATH
\nif os.path.exists(DATA_PATH):\n    df = pd.read_csv(DATA_PATH, parse_dates=['datetime', 'date'])\n    print('已加载真实数据')\nelse:\n    np.random.seed(42)\n    n = 100000\n    users = np.random.choice(range(1000, 9000), n)\n    items = np.random.choice(range(100000, 900000), n)\n    cates = np.random.choice(range(100, 1200), n)\n    behaviors = np.random.choice(['pv', 'buy', 'cart', 'fav'], n, p=[0.70, 0.05, 0.15, 0.10])\n    start = datetime(2017, 11, 25)\n    deltas = np.random.randint(0, 9*24*3600, n)\n    datetimes = [start + timedelta(seconds=int(s)) for s in deltas]\n    df = pd.DataFrame({\n        'user_id': users, 'item_id': items, 'category_id': cates,\n        'behavior_type': behaviors, 'timestamp': [int(d.timestamp()) for d in datetimes],\n        'datetime': datetimes,\n    })\n    df['date'] = df['datetime'].dt.date\n    df['hour'] = df['datetime'].dt.hour\n    df['day_of_week'] = df['datetime'].dt.dayofweek\n    df['is_weekend'] = df['day_of_week'].isin([5, 6]).astype(int)\n    def time_period(h):\n        if 0 <= h < 6: return '凌晨'\n        elif 6 <= h < 12: return '上午'\n        elif 12 <= h < 18: return '下午'\n        else: return '晚上'\n    df['time_period'] = df['hour'].apply(time_period)\n    print('已生成模拟数据')\n\ndf.head()

## 2. 构建用户-物品交互矩阵\n\n为简化计算，仅选取有购买行为的用户与商品，构建二值交互矩阵（1=购买过，0=未购买）。

In [ ]:
# 仅使用购买行为构建矩阵\nbuy_df = df[df['behavior_type'] == 'buy'][['user_id', 'item_id']].drop_duplicates()\nbuy_df['rating'] = 1\n\n# 选取交互较活跃的用户与商品（避免矩阵过于稀疏）\nuser_counts = buy_df['user_id'].value_counts()\nitem_counts = buy_df['item_id'].value_counts()\nactive_users = user_counts[user_counts >= 2].index[:500]  # 最多500个用户\nactive_items = item_counts[item_counts >= 2].index[:500]  # 最多500个商品\n\nfiltered = buy_df[buy_df['user_id'].isin(active_users) & buy_df['item_id'].isin(active_items)]\n\n# 构建透视矩阵\nuser_item_matrix = filtered.pivot_table(index='user_id', columns='item_id', values='rating', fill_value=0)\nprint('交互矩阵形状:', user_item_matrix.shape)

## 3. 计算用户相似度矩阵（余弦相似度）

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity\n\n# 余弦相似度\nuser_similarity = cosine_similarity(user_item_matrix)\nuser_similarity_df = pd.DataFrame(user_similarity, index=user_item_matrix.index, columns=user_item_matrix.index)\n\n# 可视化相似度热力图（取前30用户）\nsample_users = user_similarity_df.index[:30]\nplt.figure(figsize=(10, 8))\nsns.heatmap(user_similarity_df.loc[sample_users, sample_users], cmap='YlGnBu', square=True)\nplt.title('用户相似度热力图（余弦相似度，前30用户）', fontsize=14)\nplt.tight_layout()\nplt.savefig(f'{IMG_DIR}/04_user_similarity_heatmap.png', dpi=150, bbox_inches='tight')\nplt.show()

## 4. 基于用户的协同过滤推荐（UserCF）

In [ ]:
def recommend_for_user(user_id: int, user_item_matrix: pd.DataFrame,\n                       user_similarity_df: pd.DataFrame, top_k: int = 10, n_recommend: int = 5):\n    """\n    为指定用户生成Top-N推荐列表（UserCF）。\n    """\n    if user_id not in user_item_matrix.index:\n        return []\n    """\n    # 找到最相似的K个用户（排除自己）\n    sim_users = user_similarity_df.loc[user_id].drop(user_id).sort_values(ascending=False).head(top_k)\n\n    # 目标用户已交互的物品\n    user_items = set(user_item_matrix.columns[user_item_matrix.loc[user_id] > 0])\n\n    # 计算候选物品的加权得分\n    scores = defaultdict(float)\n    for sim_user, sim_score in sim_users.items():\n        sim_user_items = set(user_item_matrix.columns[user_item_matrix.loc[sim_user] > 0])\n        for item in sim_user_items - user_items:\n            scores[item] += sim_score\n\n    # 排序返回Top-N\n    recommended = sorted(scores.items(), key=lambda x: x[1], reverse=True)[:n_recommend]\n    return [item for item, score in recommended]\n\n# 为第一个用户生成推荐\ntarget_user = user_item_matrix.index[0]\nrecs = recommend_for_user(target_user, user_item_matrix, user_similarity_df, top_k=10, n_recommend=5)\nprint(f'为用户 {target_user} 推荐商品: {recs}')

## 5. 推荐效果评估\n\n采用留一法（Leave-One-Out）思路：对每个用户隐藏其最后一件购买商品，用其余购买记录生成推荐，检查被隐藏的商品是否出现在推荐列表中。\n评估指标：准确率（Precision@K）与召回率（Recall@K）。

In [ ]:
def evaluate_usercf(user_item_matrix: pd.DataFrame, user_similarity_df: pd.DataFrame,\n                    top_k: int = 10, n_recommend: int = 5):\n    """\n    使用留一法评估UserCF的Precision@K与Recall@K。\n    """\n    precisions = []\n    recalls = []\n    valid_users = 0\n\n    for user_id in user_item_matrix.index:\n        user_items = user_item_matrix.columns[user_item_matrix.loc[user_id] > 0].tolist()\n        if len(user_items) < 2:\n            continue\n        valid_users += 1\n        # 隐藏最后一件\n        held_out = user_items[-1]\n        train_items = user_items[:-1]\n\n        # 构造临时矩阵（仅用于该用户评估）\n        temp_matrix = user_item_matrix.copy()\n        temp_matrix.loc[user_id, held_out] = 0\n\n        # 基于临时矩阵生成推荐（简化：直接用完整相似度，但候选时排除held_out）\n        sim_users = user_similarity_df.loc[user_id].drop(user_id).sort_values(ascending=False).head(top_k)\n        scores = defaultdict(float)\n        for sim_user, sim_score in sim_users.items():\n            sim_user_items = set(user_item_matrix.columns[user_item_matrix.loc[sim_user] > 0])\n            for item in sim_user_items - set(train_items):\n                scores[item] += sim_score\n        recommended = [item for item, s in sorted(scores.items(), key=lambda x: x[1], reverse=True)[:n_recommend]]\n\n        hit = 1 if held_out in recommended else 0\n        precisions.append(hit / len(recommended) if recommended else 0)\n        recalls.append(hit / 1)  # 只隐藏1件，分母为1\n\n    return np.mean(precisions), np.mean(recalls), valid_users\n\nprecision, recall, n_users = evaluate_usercf(user_item_matrix, user_similarity_df, top_k=10, n_recommend=5)\nprint(f'评估用户数: {n_users}')\nprint(f'Precision@5: {precision:.4f}')\nprint(f'Recall@5: {recall:.4f}')

## 6. 结果可视化

In [ ]:
metrics = pd.DataFrame({\n    '指标': ['Precision@5', 'Recall@5'],\n    '数值': [precision, recall]\n})\n\nplt.figure(figsize=(8, 5))\nsns.barplot(data=metrics, x='指标', y='数值', palette='muted')\nplt.title('UserCF 推荐效果评估', fontsize=14)\nplt.ylim(0, 1)\nfor i, v in enumerate(metrics['数值']):\n    plt.text(i, v + 0.02, f'{v:.3f}', ha='center', va='bottom')\nplt.tight_layout()\nplt.savefig(f'{IMG_DIR}/04_recommendation_metrics.png', dpi=150, bbox_inches='tight')\nplt.show()

## 7. 总结\n\n1. **协同过滤原理**：通过计算用户间的相似度，将相似用户喜欢的物品推荐给目标用户。\n2. **稀疏性问题**：电商数据中用户-物品矩阵极度稀疏，实际生产中需结合矩阵分解（SVD）或深度学习模型（如DeepFM）缓解。\n3. **评估指标**：Precision@K衡量推荐列表的准确性，Recall@K衡量对潜在兴趣的覆盖度。两者需根据业务场景权衡。\n4. **改进方向**：引入多行为权重（点击1分、收藏2分、加购3分、购买5分）可提升相似度计算精度；结合ItemCF可形成混合推荐策略。